# Qwen3-14B + 自作 LoRA を vLLM バックエンドで推論

このノートブックは `src/llmlab/backends/vllm_backend.py` の API を利用して、Qwen3-14B（HF Hub）と自作 LoRA アダプタを組み合わせた推論を行う Google Colab 向けワークフローです。各セルの役割は次の通りです。

1. 環境構築（GitHub 取得 / 依存インストール）
2. パス・モデル設定
3. バックエンド API 読み込み
4. vLLM モデルロード
5. バッチ推論 + メトリクス計測
6. 対話モード（任意）
7. 後片付け


In [ ]:
# === 1. 環境構築: GitHub 取得と依存インストール（Google Colab 想定） ===
import os
import sys

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover - Colab 前提
    get_ipython = None  # type: ignore

def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython 環境が必要です。Colab で実行してください。")
    return ip

def clone_repo(url: str, target: str) -> None:
    if os.path.exists(target):
        print("既存リポジトリを再利用します:", target)
        return
    ip = _require_ipython()
    print("リポジトリをクローンします:", url)
    ip.system(f"git clone {url} {target}")

def pip_install(packages) -> None:
    packages = list(packages)
    if not packages:
        return
    ip = _require_ipython()
    quoted = " ".join(f'"{pkg}"' for pkg in packages)
    print("pip install:", packages)
    ip.run_line_magic("pip", f"install --upgrade {quoted}")

REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR  # 必要に応じて編集してください

BASE_PACKAGES = [
    "transformers>=4.56.0,<4.57.0",
    "peft>=0.17.0,<0.18.0",
    "vllm",
    "pyyaml",
]
ADDITIONAL_PACKAGES = []  # 任意の追加パッケージがあれば文字列を列挙

clone_repo(REPO_URL, REPO_DIR)
pip_install(BASE_PACKAGES + ADDITIONAL_PACKAGES)


In [ ]:
# === 2. パス・モデル定数設定 ===
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_ROOT = Path(REPO_DIR).resolve()
DATA_ROOT = Path("/content/drive/MyDrive/llm-lab-save") if IS_COLAB else Path.cwd() / "runtime"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "MODEL_NAME": "Qwen/Qwen3-14B",
    "LORA_DIR": DATA_ROOT / "models" / "adapters" / "qwen3-14b-lora-ojousama",
    "TENSOR_PARALLEL_SIZE": 1,
    "DTYPE": "auto",
    "MAX_MODEL_LEN": 4096,
    "GPU_MEMORY_UTILIZATION": 0.90,
    "DOWNLOAD_DIR": DATA_ROOT / "model_cache",
    "QUANTIZATION": "none",
    "MERGE_LORA": False,
}

GENERATION = {
    "MAX_NEW_TOKENS": 256,
    "TEMPERATURE": 0.7,
    "TOP_P": 0.9,
    "REPETITION_PENALTY": 1.05,
    "STOP": ["\\nUser:"],
}

CHAT = {
    "SYSTEM_PROMPT": "あなたは Qwen3-14B ベースのアシスタントです。",
    "STOP_PHRASES": ["/exit", ":q"],
    "EXIT_COMMAND": "/exit",
}

CONFIG["LORA_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["DOWNLOAD_DIR"].mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_NAME: {CONFIG['MODEL_NAME']}")
print(f"LORA_DIR: {CONFIG['LORA_DIR']}")
print("必要に応じて上記定数を編集してください。")


In [ ]:
# === 3. Python パス調整とバックエンド API の読み込み ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.llmlab.backends.vllm_backend import (
    ModelBundle,
    chat_loop,
    free_model,
    load_model,
    profile_generation,
)

print("バックエンド API をロードしました。")


In [ ]:
# === 4. vLLM モデルロード ===
lora_dir = CONFIG["LORA_DIR"]
lora_path = str(lora_dir) if lora_dir.exists() and any(lora_dir.glob("**/*")) else None

cfg = {
    "model_name": CONFIG["MODEL_NAME"],
    "tensor_parallel_size": CONFIG["TENSOR_PARALLEL_SIZE"],
    "dtype": CONFIG["DTYPE"],
    "max_model_len": CONFIG["MAX_MODEL_LEN"],
    "gpu_memory_utilization": CONFIG["GPU_MEMORY_UTILIZATION"],
    "download_dir": str(CONFIG["DOWNLOAD_DIR"]),
    "quantization": CONFIG["QUANTIZATION"],
    "lora_path": lora_path,
    "merge_lora": CONFIG["MERGE_LORA"],
}

print("ロード設定:", cfg)
bundle: ModelBundle = load_model(cfg)
print("モデルロード完了。トークナイザ取得可否:", bool(bundle["tok"]))
print("ロード時間(秒):", bundle["cfg"].get("_timings", {}))


In [ ]:
# === 5. バッチ推論 & メトリクス計測 ===
import pandas as pd

prompts = [
    "以下の仕様を要約してください:\n- 多段 LoRA でチューニングした Qwen3-14B\n- 連携するエッジ端末向けアプリへの適用",
    "LoRA 適用済みモデルとして、顧客向け FAQ 生成ボットの導入メリットを3点で説明してください。",
]

outputs, metrics = profile_generation(
    bundle,
    prompts,
    max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
    temperature=GENERATION["TEMPERATURE"],
    top_p=GENERATION["TOP_P"],
    repetition_penalty=GENERATION["REPETITION_PENALTY"],
    stop=GENERATION["STOP"],
)

display(pd.DataFrame({"prompt": prompts, "output": outputs}))
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


In [ ]:
# === 6. 対話モード（任意） ===
print(f"対話を開始します。終了コマンド: {CHAT['EXIT_COMMAND']}")
try:
    chat_loop(
        bundle,
        system_prompt=CHAT["SYSTEM_PROMPT"],
        stop_phrases=list({*CHAT["STOP_PHRASES"], CHAT["EXIT_COMMAND"]}),
        max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
        temperature=GENERATION["TEMPERATURE"],
        top_p=GENERATION["TOP_P"],
        repetition_penalty=GENERATION["REPETITION_PENALTY"],
    )
finally:
    print("対話モードを終了しました。")


In [ ]:
# === 7. 後片付け ===
free_model(bundle)
print("メモリを解放しました。")
